In [2]:
import pandas as pd
df = pd.read_csv("online_retail.csv")

print("Raw rows:", len(df))
df = df.dropna(subset=["Customer ID"])                        # drop rows with no customer
df = df[~df["Invoice"].astype(str).str.startswith("C")]    # drop cancellations
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]        # drop bad rows
df["revenue"] = df["Quantity"] * df["Price"]
print("Clean rows:", len(df), "| Customers:", df["Customer ID"].nunique())

Raw rows: 1067371
Clean rows: 805549 | Customers: 5878


In [4]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
snapshot = df["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = df.groupby("Customer ID").agg(
    recency=("InvoiceDate", lambda x: (snapshot - x.max()).days),
    frequency=("Invoice", "nunique"),
    monetary=("revenue", "sum"),
)

rfm["R"] = pd.qcut(rfm["recency"], 4, labels=[4, 3, 2, 1]).astype(int)
rfm["F"] = pd.qcut(rfm["frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["M"] = pd.qcut(rfm["monetary"], 4, labels=[1, 2, 3, 4]).astype(int)
rfm["score"] = rfm["R"] + rfm["F"] + rfm["M"]

def segment(s):
    if s >= 10: return "Champions"
    if s >= 7:  return "Loyal"
    if s >= 5:  return "At risk"
    return "Lost"

rfm["segment"] = rfm["score"].apply(segment)
summary = rfm.groupby("segment").agg(
    customers=("monetary", "count"),
    revenue=("monetary", "sum"),
).round(0)
summary["revenue_share_%"] = (summary["revenue"] / summary["revenue"].sum() * 100).round(1)
summary

,customers,revenue,revenue_share_%
segment,,,
At risk,1212,670178.0,3.8
Champions,1738,14328763.0,80.8
Lost,1147,261402.0,1.5
Loyal,1781,2483086.0,14.0


In [5]:
df["month"] = df["InvoiceDate"].dt.to_period("M")
df["cohort"] = df.groupby("Customer ID")["InvoiceDate"].transform("min").dt.to_period("M")
df["months_since"] = (df["month"] - df["cohort"]).apply(lambda x: x.n)

cohort = df.groupby(["cohort", "months_since"])["Customer ID"].nunique().unstack()
retention = (cohort.div(cohort[0], axis=0) * 100).round(1)
retention.iloc[:, :7]    # first 7 months

months_since,0,1,2,3,4,5,6
cohort,,,,,,,
2009-12,100.0,35.3,33.4,42.5,38.0,35.9,37.7
2010-01,100.0,20.6,31.1,30.5,26.4,30.0,25.8
2010-02,100.0,23.8,22.5,29.1,24.6,20.1,19.3
2010-03,100.0,19.0,23.0,24.2,23.3,20.3,24.6
2010-04,100.0,19.4,19.4,16.3,18.4,22.4,27.6
2010-05,100.0,15.7,16.9,17.3,17.7,25.6,21.3
2010-06,100.0,17.4,18.9,20.4,23.0,28.5,12.6
2010-07,100.0,15.6,18.3,29.6,29.0,14.0,11.3
2010-08,100.0,20.4,29.6,32.1,17.3,11.7,9.9


In [6]:
top20 = rfm.sort_values("monetary", ascending=False).head(int(len(rfm) * 0.2))
print(f"Top 20% of customers = {top20['monetary'].sum() / rfm['monetary'].sum():.0%} of revenue")

Top 20% of customers = 77% of revenue
